In [ ]:
# ============================================================
# CELL 1 — Configure AWS and Verify Connections
# ============================================================
import os, subprocess
from google.colab import userdata

# Load credentials from Colab Secrets
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
DB_PASSWORD = userdata.get('DB_PASSWORD')

DB_HOST          = 'geoai-mlops-p1-postgres.cqbao8c24eu5.us-east-1.rds.amazonaws.com'
DB_USER          = 'geoai_admin'
DB_NAME_FEATURES = 'geoai_features'
S3_BUCKET        = 'geoai-mlops-p1-data-288528696055'
MLFLOW_URL       = 'http://3.91.55.220:5000'

# Set in os.environ for all subsequent cells
os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

# Install boto3 in system Python for verification
os.system('pip install -q boto3')

# Verify S3
import boto3
s3   = boto3.client('s3', region_name='us-east-1')
resp = s3.list_objects_v2(
    Bucket=S3_BUCKET, Prefix='processed/patches/', MaxKeys=1)
print(f'✅ S3 connected — objects: {resp["KeyCount"]}')

# Verify GPU
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f'✅ GPU: {r.stdout.strip()}')
else:
    print('⚠️  No GPU detected')

print()
print(f'   AWS_KEY:   ...{AWS_KEY[-4:]}')
print(f'   S3_BUCKET: {S3_BUCKET}')
print(f'   DB_HOST:   {DB_HOST}')
print(f'   MLflow:    {MLFLOW_URL}')
print()
print('✅ Cell 1 complete — AWS configured')

In [ ]:
# ============================================================
# CELL 2 — Setup Virtual Environment
# ============================================================
import subprocess, os

VENV_PYTHON = '/content/venv/bin/python3'
VENV_PIP    = '/content/venv/bin/pip'

# System dependencies
os.system('apt-get install -y libgeos-dev libgdal-dev -q')
os.environ["MPLBACKEND"] = "agg"

# Create venv
os.system('python3 -m venv /content/venv')
os.system('curl -sS https://bootstrap.pypa.io/get-pip.py | /content/venv/bin/python3')

def install(packages):
    r = subprocess.run(
        [VENV_PIP, 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'Error: {r.stderr[-500:]}')
        return False
    return True

print('Step 1: terratorch (first — owns its dependencies)...')
install(['terratorch'])

print('Step 2: torch (force reinstall after terratorch)...')
install(['--force-reinstall', 'torch==2.2.0', 'torchvision==0.17.0'])

print('Step 3: dependencies...')
install([
    'transformers==4.40.0', 'timm', 'einops',
    'mlflow==2.14.3', 'psycopg2-binary', 'boto3',
    'rasterio', 'scipy', 'pystac-client',
    'huggingface_hub==0.20.3', 'awscli', 'pyyaml',
    'pandas', 'tqdm',
])

print('Step 4: pin all critical versions last...')
install([
    'numpy==1.26.4',
    'huggingface-hub==0.20.3',
    'protobuf==3.20.3',
    'setuptools==69.5.1',
])

os.system('pip install -q boto3 rasterio pandas')
print('✅ boto3 + rasterio + pandas installed in system Python')

r = subprocess.run([VENV_PYTHON, '-c', '''
import os; os.environ["MPLBACKEND"] = "agg"
import numpy as np, torch, pkg_resources
import google.protobuf
from terratorch.registry import BACKBONE_REGISTRY
print(f"numpy:      {np.__version__}")
print(f"torch:      {torch.__version__}")
print(f"protobuf:   {google.protobuf.__version__}")
print(f"setuptools: {pkg_resources.get_distribution('setuptools').version}")
print("terratorch: OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-200:])
print('✅ Cell 2 complete — venv ready')


In [ ]:
# ============================================================
# CELL 3 — Download Repo + Pairs + Stats + Features Cache
# Pairs, stats and features saved to Google Drive (skip if exist)
# ============================================================
import subprocess, os, boto3, tarfile
from google.colab import drive, userdata

VENV_PYTHON = '/content/venv/bin/python3'
S3_BUCKET   = 'geoai-mlops-p1-data-288528696055'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

# Mount Google Drive
drive.mount('/gdrive')

# Create Google Drive directories
os.makedirs('/gdrive/MyDrive/geoai_mlops/pairs',             exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/stats',             exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/patches',           exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/features',          exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/checkpoints/local', exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/checkpoints/best',  exist_ok=True)
print('✅ Google Drive directories ready')

s3 = boto3.client('s3', region_name='us-east-1')

# Download and extract repo to /content (always fresh — small file)
print('\nDownloading repo from S3...')
s3.download_file(S3_BUCKET, 'repo/geoai-mlops-p1-colab-fixes.tar.gz',
                 '/tmp/geoai-mlops-p1.tar.gz')
size = os.path.getsize('/tmp/geoai-mlops-p1.tar.gz')
print(f'Downloaded: {size/1e6:.1f}MB')

with tarfile.open('/tmp/geoai-mlops-p1.tar.gz', 'r:gz') as tar:
    tar.extractall('/content/')
print('✅ Repo extracted')
print('   Training files:', os.listdir('/content/geoai-mlops-p1/src/training/'))

# Create /content data dirs
os.makedirs('/content/geoai-mlops-p1/data/pairs',             exist_ok=True)
os.makedirs('/content/geoai-mlops-p1/data/stats',             exist_ok=True)
os.makedirs('/content/geoai-mlops-p1/data/checkpoints/local', exist_ok=True)

# Download pairs → Google Drive (skip if already exist)
print('\nDownloading pairs to Google Drive...')
for f in ['train_pairs.csv', 'val_pairs.csv', 'test_pairs.csv']:
    path = f'/gdrive/MyDrive/geoai_mlops/pairs/{f}'
    if os.path.exists(path):
        print(f'  ✅ {f} already exists — skipping')
    else:
        s3.download_file(S3_BUCKET, f'training/pairs/{f}', path)
        print(f'  ✅ {f} downloaded')

# Download stats → Google Drive (skip if already exist)
print('\nDownloading stats to Google Drive...')
for f in ['tabular_stats.json', 'band_stats.json',
          'day_gap_stats.json', 'stats_summary.json']:
    path = f'/gdrive/MyDrive/geoai_mlops/stats/{f}'
    if os.path.exists(path):
        print(f'  ✅ {f} already exists — skipping')
    else:
        s3.download_file(S3_BUCKET, f'training/stats/{f}', path)
        print(f'  ✅ {f} downloaded')

# Download features cache → Google Drive (skip if exists)
print('\nDownloading features cache to Google Drive...')
path = '/gdrive/MyDrive/geoai_mlops/features/features_cache.parquet'
if os.path.exists(path):
    size = os.path.getsize(path)
    print(f'  ✅ features_cache.parquet already exists ({size/1e6:.1f}MB) — skipping')
else:
    s3.download_file(
        S3_BUCKET,
        'training/features/features_cache.parquet',
        path
    )
    size = os.path.getsize(path)
    print(f'  ✅ features_cache.parquet downloaded ({size/1e6:.1f}MB)')

print()
print('✅ Cell 3 complete')



In [ ]:
# ============================================================
# CELL 6 — Verify Prithvi-EO-1.0-100M Loads Correctly
# Confirms TerraTorch + encoder working before training
# ============================================================
import subprocess
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

r = subprocess.run([VENV_PYTHON, '-c', f'''
import os
os.environ["MPLBACKEND"]            = "agg"
os.environ["AWS_ACCESS_KEY_ID"]     = "{AWS_KEY}"
os.environ["AWS_SECRET_ACCESS_KEY"] = "{AWS_SECRET}"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import torch
from terratorch.registry import BACKBONE_REGISTRY

print("Loading Prithvi-EO-1.0-100M via TerraTorch...")
encoder = BACKBONE_REGISTRY.build(
    "prithvi_eo_v1_100",
    pretrained=True,
    num_frames=1,
    in_chans=6,
)
params = sum(p.numel() for p in encoder.parameters())
print(f"✅ Encoder loaded: {{params/1e6:.0f}}M parameters")

# Test forward pass
x = torch.zeros(1, 6, 224, 224)
with torch.no_grad():
    out = encoder(x)

# Extract embedding — last layer, mean pool patch tokens
embedding = out[-1][:, 1:, :].mean(dim=1)
print(f"✅ Embedding shape: {{embedding.shape}}")
assert embedding.shape == torch.Size([1, 768]), f"Expected [1,768] got {{embedding.shape}}"

# Test on GPU if available
if torch.cuda.is_available():
    encoder = encoder.cuda()
    with torch.no_grad():
        out_gpu = encoder(x.cuda())
    emb_gpu = out_gpu[-1][:, 1:, :].mean(dim=1)
    print(f"✅ GPU forward pass: {{emb_gpu.shape}}")
    print(f"   GPU: {{torch.cuda.get_device_name(0)}}")
else:
    print("⚠️  No GPU detected — check runtime type")

del encoder
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("✅ Cell 6 complete — Prithvi ready for training")
'''], capture_output=True, text=True)

print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-300:])



In [ ]:
# ============================================================
# CELL 9 — Munich Change Detection Inference
# Runs inference on Munich Sentinel-2 2020→2023
# Outputs: CSV, GeoJSON, GeoTIFF, summary
# ============================================================
import subprocess, os, json, glob
from google.colab import userdata, drive

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

# Mount GDrive
drive.mount('/gdrive')

# Output directory
OUTPUT_DIR  = '/gdrive/MyDrive/geoai_mlops/inference/munich_2020_2023'
TMP_DIR     = '/content/munich_scenes'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR,    exist_ok=True)

# Best model checkpoint
BEST_DIR    = '/gdrive/MyDrive/geoai_mlops/checkpoints/best'
pattern     = os.path.join(BEST_DIR, 'best_model_chunk6_r2_*.pt')
candidates  = glob.glob(pattern)

if not candidates:
    # Fallback — pick highest r2 across all chunks
    pattern   = os.path.join(BEST_DIR, 'best_model_chunk*_r2_*.pt')
    candidates = glob.glob(pattern)

if not candidates:
    raise FileNotFoundError(f'No model checkpoint found in {BEST_DIR}')

model_path = max(
    candidates,
    key=lambda x: float(x.split('_r2_')[1].replace('.pt', ''))
)
print(f'✅ Using model: {os.path.basename(model_path)}')

env = {
    **os.environ,
    'MPLBACKEND':            'agg',
    'AWS_ACCESS_KEY_ID':     AWS_KEY,
    'AWS_SECRET_ACCESS_KEY': AWS_SECRET,
    'AWS_DEFAULT_REGION':    'us-east-1',
}

# Install geopandas in venv if not already
print('Checking geopandas...')
r = subprocess.run(
    [VENV_PYTHON, '-c', 'import geopandas; print(geopandas.__version__)'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Installing geopandas...')
    subprocess.run(
        [VENV_PYTHON, '-m', 'pip', 'install', '-q',
         'geopandas', 'shapely', 'fiona'],
        capture_output=True
    )
    print('✅ geopandas installed')
else:
    print(f'✅ geopandas {r.stdout.strip()} already installed')

# Build inference command
cmd = [
    VENV_PYTHON,
    'src/inference/inference_munich.py',
    '--model',      model_path,
    '--output',     OUTPUT_DIR,
    '--tmp',        TMP_DIR,
    '--batch-size', '8',
]

print(f'\n{"═"*60}')
print(f'  MUNICH INFERENCE')
print(f'  Model:  {os.path.basename(model_path)}')
print(f'  Output: {OUTPUT_DIR}')
print(f'  T1: 2020-07-07 → T2: 2023-07-07 (1096 days)')
print(f'{"═"*60}\n')

# Stream output
process = subprocess.Popen(
    cmd,
    cwd='/content/geoai-mlops-p1',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\nReturn code: {process.returncode}')

if process.returncode == 0:
    # Show summary
    summary_path = os.path.join(OUTPUT_DIR, 'munich_summary.json')
    if os.path.exists(summary_path):
        summary = json.load(open(summary_path))
        print(f'\n{"═"*60}')
        print(f'  MUNICH INFERENCE COMPLETE')
        print(f'  Total patches:  {summary["total_patches"]:,}')
        print(f'  Change score:   mean={summary["change_score"]["mean"]:.4f}')
        print(f'  Change classes:')
        for cls, pct in summary['change_class_pct'].items():
            print(f'    {cls:<20}: {pct:.1f}%')
        print(f'{"═"*60}')

    # List output files
    print(f'\nOutput files:')
    for f in sorted(os.listdir(OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
        print(f'  {f:<45} {size/1e6:.1f}MB')